# Gold — run every poc-style fact (task `gold_facts_poc`)

Runs the per-table fact notebooks (poc method: `add_checksums` + `full_refresh` / upsert on `primary_checksum`) with `dbutils.notebook.run`. Test tables only (`*_poc`): nothing reads them; the live facts (`facts.ipynb`, `restaurants.ipynb`, `place_rating.ipynb`) and the agg tables are unchanged.

| Order | Notebook | Needs |
|---|---|---|
| 1 | fact_video_snapshot_poc | stg_videos, dim_video_scd1, dim_channel_scd1 |
| 2 | fact_channel_snapshot_poc | stg_channels, dim_channel_scd1 |
| 3 | fact_search_hit_poc | stg_search_results, dim_keyword_scd1, dim_video_scd1, dim_channel_scd1 |
| 4 | fact_video_restaurant_mention_poc | stg_restaurant_mentions, stg_restaurant_geocode, dim_video_scd1 |
| 5 | fact_place_rating_snapshot_poc | stg_restaurant_place_details, stg_places, dim_restaurant |

A table that does not exist yet is loaded with `load_type = full` first (like poc `update_load_type`). A failing table does not stop the others; the task fails at the end if any failed.

In [ ]:
for w, default in (("silver", "prod.silver"), ("gold", "prod.gold"), ("load_type", "incremental"), ("from_date", ""), ("to_date", "")):
    dbutils.widgets.text(w, default)
args = {w: dbutils.widgets.get(w) for w in ("silver", "gold", "load_type", "from_date", "to_date")}
ORDER = ["fact_video_snapshot_poc", "fact_channel_snapshot_poc", "fact_search_hit_poc",
         "fact_video_restaurant_mention_poc", "fact_place_rating_snapshot_poc"]
failed = {}
for name in ORDER:
    try:
        # first load of a table = full (poc: load_type full once, then update_load_type switches it to incremental)
        run_args = dict(args, load_type="full") if not spark.catalog.tableExists(f"{args['gold']}.{name}") else args
        out = dbutils.notebook.run(f"./{name}", 1800, run_args)
        print(f"✅ {name} ({run_args['load_type']})" + (f": {out}" if out else ""))
    except Exception as e:
        failed[name] = str(e)[:500]
        print(f"❌ {name}: {str(e)[:500]}")
if failed:
    raise RuntimeError(f"{len(failed)} of {len(ORDER)} poc facts failed: {list(failed)}")
print(f"--- {len(ORDER)} poc facts done ({args['load_type']}) ---")